In [ ]:
from hcc_release_paths import project_path as _hcc_path
# ==========================================================
# Step14.1
# Prepare expression matrix and final subtype labels
# ==========================================================

import os
import pandas as pd
import numpy as np

# ----------------------------------------------------------
# Paths
# ----------------------------------------------------------

step14_dir = _hcc_path('LIHC_RNA_Step14_Pathway')
input_dir = os.path.join(step14_dir, "Input")
results_dir = os.path.join(step14_dir, "Results")
figures_dir = os.path.join(step14_dir, "Figures")

for d in [
    step14_dir,
    input_dir,
    results_dir,
    figures_dir
]:
    os.makedirs(d, exist_ok=True)


# ----------------------------------------------------------
# 1. Load expression matrix
# gene × sample
# ----------------------------------------------------------

expression_file = (
    r"E:\TCGA\LIHC_RNA_Step11_Immune\Input"
    r"\LIHC_Expression_365_log2CPM.tsv"
)

expression_all = pd.read_csv(
    expression_file,
    sep="\t",
    index_col=0
)

print("Expression:", expression_all.shape)


# ----------------------------------------------------------
# 2. Load retained subtype solutions
# ----------------------------------------------------------

subtype_file = (
    r"E:\TCGA\LIHC_RNA_Step13_Molecular_Subtyping\Results"
    r"\LIHC_Retained_3_Molecular_Subtype_Solutions.csv"
)

Subtype_Solutions = pd.read_csv(
    subtype_file,
    index_col=0
)

print("Subtype solutions:", Subtype_Solutions.shape)
print(Subtype_Solutions.columns.tolist())

In [ ]:
# ==========================================================
# Step14.1.2
# Final primary subtype
# ==========================================================

subtype = (
    Subtype_Solutions[
        "D_Top30_each_K2_Name"
    ]
    .copy()
)

print(
    subtype.value_counts()
)

In [ ]:
# ==========================================================
# Step14.1.3
# Match samples
# ==========================================================

common_samples = (
    expression_all.columns
    .intersection(
        subtype.index
    )
)

expression_step14 = (
    expression_all[
        common_samples
    ]
    .copy()
)

subtype_step14 = (
    subtype.loc[
        common_samples
    ]
    .copy()
)

print(
    "Matched expression:",
    expression_step14.shape
)

print(
    "\nSubtype counts:"
)

print(
    subtype_step14.value_counts()
)

In [ ]:
# ==========================================================
# Save Step14 input
# ==========================================================

expression_step14.to_csv(
    os.path.join(
        input_dir,
        "LIHC_Expression_365_log2CPM.tsv"
    ),
    sep="\t"
)

subtype_step14.to_frame(
    "Molecular_Subtype"
).to_csv(
    os.path.join(
        input_dir,
        "LIHC_D_Top30_K2_Subtype.csv"
    )
)

print("Step14 input saved.")

In [ ]:
# ==========================================================
# Step14.2.1
# Build ranked gene list
# Proliferative vs Metabolic
# ==========================================================

from scipy.stats import ttest_ind
import pandas as pd
import numpy as np

pro_samples = subtype_step14[
    subtype_step14 == "Proliferative"
].index

met_samples = subtype_step14[
    subtype_step14 == "Metabolic"
].index


rank_rows = []

for gene in expression_step14.index:

    x_pro = expression_step14.loc[
        gene,
        pro_samples
    ].values

    x_met = expression_step14.loc[
        gene,
        met_samples
    ].values

    stat, p = ttest_ind(
        x_pro,
        x_met,
        equal_var=False,
        nan_policy="omit"
    )

    rank_rows.append({
        "Gene": gene,
        "T_stat": stat,
        "P_value": p
    })


gsea_rank = pd.DataFrame(
    rank_rows
)

gsea_rank = (
    gsea_rank
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .dropna(
        subset=["T_stat"]
    )
    .sort_values(
        "T_stat",
        ascending=False
    )
)

print(gsea_rank.shape)

gsea_rank.head(10)

In [ ]:
gsea_rank.to_csv(
    os.path.join(
        input_dir,
        "LIHC_Proliferative_vs_Metabolic_GSEA_Ranking.csv"
    ),
    index=False
)

In [ ]:
# ==========================================================
# Step14.2.2
# Hallmark preranked GSEA
# ==========================================================

import gseapy as gp

pre_res = gp.prerank(
    rnk=gsea_rank[
        ["Gene", "T_stat"]
    ],

    gene_sets="MSigDB_Hallmark_2020",

    permutation_num=1000,

    min_size=10,
    max_size=500,

    seed=42,

    outdir=None,

    verbose=True
)

print("GSEA finished.")

In [ ]:
# ==========================================================
# Step14.2.3
# Extract GSEA results
# ==========================================================

gsea_results = (
    pre_res.res2d
    .copy()
    .reset_index(drop=True)
)

print(gsea_results.columns.tolist())

gsea_results.head()

In [ ]:
# ==========================================================
# Add direction
# ==========================================================

gsea_results["Direction"] = np.where(
    gsea_results["NES"] > 0,
    "Proliferative",
    "Metabolic"
)

gsea_results["Significant"] = (
    gsea_results["FDR q-val"] < 0.05
)

gsea_sig = (
    gsea_results[
        gsea_results["Significant"]
    ]
    .copy()
)

gsea_sig = (
    gsea_sig
    .sort_values(
        "NES",
        ascending=False
    )
)

print(
    "Significant Hallmark pathways:",
    len(gsea_sig)
)

display(
    gsea_sig[
        [
            "Term",
            "NES",
            "NOM p-val",
            "FDR q-val",
            "Direction"
        ]
    ]
)

In [ ]:
gsea_results.to_csv(
    os.path.join(
        results_dir,
        "Hallmark_GSEA_All.csv"
    ),
    index=False
)

gsea_sig.to_csv(
    os.path.join(
        results_dir,
        "Hallmark_GSEA_Significant.csv"
    ),
    index=False
)

In [ ]:
# ==========================================================
# Step14.3.1
# Hallmark GSVA
# gene × sample expression matrix
# ==========================================================

import gseapy as gp
import pandas as pd
import numpy as np
import os

gsva_res = gp.gsva(
    data=expression_step14,
    gene_sets="MSigDB_Hallmark_2020",

    min_size=10,
    max_size=500,

    kcdf="Gaussian",

    outdir=None,

    verbose=True
)

print("GSVA finished.")

In [ ]:
print(type(gsva_res))

print(gsva_res.res2d.shape)

print(gsva_res.res2d.columns.tolist())

gsva_res.res2d.head()

In [ ]:
# ==========================================================
# Convert GSVA long format to pathway × sample matrix
# ==========================================================

gsva_long = gsva_res.res2d.copy()

GSVA_matrix = gsva_long.pivot(
    index="Term",
    columns="Name",
    values="ES"
)

GSVA_matrix = GSVA_matrix.astype(float)

print(
    "GSVA matrix:",
    GSVA_matrix.shape
)

GSVA_matrix.iloc[:5, :5]

In [ ]:
GSVA_matrix.to_csv(
    os.path.join(
        results_dir,
        "Hallmark_GSVA_Score_Matrix.csv"
    )
)

print("Saved.")

In [ ]:
# ==========================================================
# Step14.3.3
# Compare GSVA scores:
# Metabolic vs Proliferative
# ==========================================================

from scipy.stats import mannwhitneyu

gsva_compare_results = []


for pathway in GSVA_matrix.index:

    pro_scores = (
        GSVA_matrix.loc[
            pathway,
            pro_samples
        ]
        .astype(float)
        .dropna()
    )

    met_scores = (
        GSVA_matrix.loc[
            pathway,
            met_samples
        ]
        .astype(float)
        .dropna()
    )


    U, p = mannwhitneyu(
        pro_scores,
        met_scores,
        alternative="two-sided"
    )


    # Effect > 0:
    # Proliferative higher
    effect = (
        2 * U /
        (
            len(pro_scores)
            *
            len(met_scores)
        )
        - 1
    )


    gsva_compare_results.append({

        "Pathway":
            pathway,

        "Proliferative_median":
            pro_scores.median(),

        "Metabolic_median":
            met_scores.median(),

        "Median_difference_Pro_minus_Met":
            pro_scores.median()
            -
            met_scores.median(),

        "Effect_Pro_vs_Met":
            effect,

        "P_value":
            p

    })


GSVA_Subtype_Comparison = pd.DataFrame(
    gsva_compare_results
)

GSVA_Subtype_Comparison.head()

In [ ]:
# ==========================================================
# BH-FDR correction
# ==========================================================

from statsmodels.stats.multitest import multipletests


GSVA_Subtype_Comparison["FDR"] = multipletests(
    GSVA_Subtype_Comparison["P_value"],
    method="fdr_bh"
)[1]


GSVA_Subtype_Comparison["Direction"] = np.where(
    GSVA_Subtype_Comparison[
        "Effect_Pro_vs_Met"
    ] > 0,
    "Proliferative",
    "Metabolic"
)


GSVA_Subtype_Comparison["Abs_effect"] = (
    GSVA_Subtype_Comparison[
        "Effect_Pro_vs_Met"
    ].abs()
)


GSVA_Subtype_Comparison = (
    GSVA_Subtype_Comparison
    .sort_values(
        "Abs_effect",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


GSVA_Subtype_Comparison.head(20)

In [ ]:
GSVA_Subtype_Comparison.to_csv(
    os.path.join(
        results_dir,
        "Hallmark_GSVA_Subtype_Comparison.csv"
    ),
    index=False
)

In [ ]:
GSVA_sig = (
    GSVA_Subtype_Comparison[
        GSVA_Subtype_Comparison[
            "FDR"
        ] < 0.05
    ]
    .copy()
)


print(
    "Significant GSVA pathways:",
    len(GSVA_sig)
)


display(
    GSVA_sig[
        [
            "Pathway",
            "Proliferative_median",
            "Metabolic_median",
            "Effect_Pro_vs_Met",
            "FDR",
            "Direction"
        ]
    ].head(25)
)

In [ ]:
# ==========================================================
# Step14.4.1
# Select representative Hallmark pathways
# ==========================================================

selected_pathways = [

    # Metabolic subtype
    "Bile Acid Metabolism",
    "Xenobiotic Metabolism",
    "Fatty Acid Metabolism",
    "Coagulation",
    "Pperoxisome",

    # Proliferative subtype
    "G2-M Checkpoint",
    "E2F Targets",
    "Myc Targets V1",
    "DNA Repair",
    "Mitotic Spindle",
    "PI3K/AKT/mTOR Signaling"
]


selected_pathways = [
    p for p in selected_pathways
    if p in GSVA_matrix.index
]

print(selected_pathways)

In [ ]:
# ==========================================================
# Sort patients by molecular subtype
# ==========================================================

sample_annotation = pd.DataFrame({
    "Subtype": subtype_step14
})

sample_annotation = (
    sample_annotation
    .sort_values(
        "Subtype",
        key=lambda x: x.map({
            "Metabolic": 0,
            "Proliferative": 1
        })
    )
)

sample_order = sample_annotation.index.tolist()

heatmap_data = GSVA_matrix.loc[
    selected_pathways,
    sample_order
].copy()

print(heatmap_data.shape)

In [ ]:
# ==========================================================
# Row-wise Z-score
# ==========================================================

heatmap_z = heatmap_data.apply(
    lambda x: (
        x - x.mean()
    ) / x.std(),
    axis=1
)

# 美化名称
heatmap_z.index = (
    heatmap_z.index
    .str.replace(
        "Pperoxisome",
        "Peroxisome",
        regex=False
    )
)

In [ ]:
# ==========================================================
# Hallmark GSVA heatmap
# ==========================================================

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

n_metabolic = (
    subtype_step14
    == "Metabolic"
).sum()

print(
    "Metabolic:",
    n_metabolic
)

print(
    "Proliferative:",
    len(subtype_step14) - n_metabolic
)

In [ ]:
fig, ax = plt.subplots(
    figsize=(12, 6.5)
)

sns.heatmap(
    heatmap_z,
    cmap="RdBu_r",
    center=0,
    xticklabels=False,
    yticklabels=True,
    cbar_kws={
        "label": "GSVA Z-score"
    },
    ax=ax
)

# 两个亚型分界线
ax.axvline(
    n_metabolic,
    linewidth=2
)

ax.set_xlabel(
    "Patients"
)

ax.set_ylabel(
    "Hallmark pathways"
)

ax.set_title(
    "Hallmark Pathway Activity Across Molecular Subtypes",
    fontsize=14
)

# 标记两个亚型位置
ax.text(
    n_metabolic / 2,
    -0.7,
    "Metabolic",
    ha="center",
    va="center",
    fontsize=11
)

ax.text(
    n_metabolic
    +
    (
        len(sample_order)
        -
        n_metabolic
    ) / 2,
    -0.7,
    "Proliferative",
    ha="center",
    va="center",
    fontsize=11
)

fig.tight_layout()

fig.savefig(
    os.path.join(
        figures_dir,
        "Hallmark_GSVA_Subtype_Heatmap.png"
    ),
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    os.path.join(
        figures_dir,
        "Hallmark_GSVA_Subtype_Heatmap.svg"
    ),
    bbox_inches="tight"
)

plt.show()

plt.close(fig)

In [ ]:
# ==========================================================
# Step14.4.5
# Heatmap of ALL Hallmark pathways
# ==========================================================

import matplotlib.pyplot as plt
import seaborn as sns

# ----------------------------------------------------------
# 1. 按亚型排序患者
# ----------------------------------------------------------

sample_annotation = pd.DataFrame({
    "Subtype": subtype_step14
})

sample_annotation = (
    sample_annotation
    .sort_values(
        "Subtype",
        key=lambda x: x.map({
            "Metabolic": 0,
            "Proliferative": 1
        })
    )
)

sample_order = sample_annotation.index.tolist()


# ----------------------------------------------------------
# 2. 全部 Hallmark pathway
# ----------------------------------------------------------

heatmap_all = GSVA_matrix.loc[
    :,
    sample_order
].copy()


# ----------------------------------------------------------
# 3. 每条 pathway 做 row-wise Z-score
# ----------------------------------------------------------

heatmap_all_z = heatmap_all.apply(
    lambda x: (
        x - x.mean()
    ) / x.std(),
    axis=1
)


# 修正显示名称
heatmap_all_z.index = (
    heatmap_all_z.index
    .str.replace(
        "Pperoxisome",
        "Peroxisome",
        regex=False
    )
)


print(
    "Heatmap matrix:",
    heatmap_all_z.shape
)

In [ ]:
# ==========================================================
# 4. Plot ALL Hallmark GSVA heatmap
# ==========================================================

n_metabolic = (
    subtype_step14 == "Metabolic"
).sum()


fig, ax = plt.subplots(
    figsize=(14, 15)
)


sns.heatmap(
    heatmap_all_z,

    cmap="RdBu_r",
    center=0,

    xticklabels=False,
    yticklabels=True,

    cbar_kws={
        "label": "GSVA Z-score",
        "shrink": 0.6
    },

    ax=ax
)


# ----------------------------------------------------------
# 分型分界线
# ----------------------------------------------------------

ax.axvline(
    n_metabolic,
    linewidth=2
)


# ----------------------------------------------------------
# 标题和坐标轴
# ----------------------------------------------------------

ax.set_title(
    "Hallmark GSVA Landscape Across Molecular Subtypes",
    fontsize=15,
    pad=25
)

ax.set_xlabel(
    "Patients"
)

ax.set_ylabel(
    "Hallmark pathways"
)


# ----------------------------------------------------------
# 亚型名称
# ----------------------------------------------------------

ax.text(
    n_metabolic / 2,
    -1.2,
    "Metabolic",
    ha="center",
    va="center",
    fontsize=12
)

ax.text(
    n_metabolic
    +
    (
        len(sample_order)
        -
        n_metabolic
    ) / 2,
    -1.2,
    "Proliferative",
    ha="center",
    va="center",
    fontsize=12
)


fig.tight_layout()


# ----------------------------------------------------------
# Save BEFORE show
# ----------------------------------------------------------

fig.savefig(
    os.path.join(
        figures_dir,
        "Hallmark_GSVA_All_Pathways_Heatmap.png"
    ),
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    os.path.join(
        figures_dir,
        "Hallmark_GSVA_All_Pathways_Heatmap.svg"
    ),
    bbox_inches="tight"
)


plt.show()

plt.close(fig)

In [ ]:
# ==========================================================
# Step14.4.6
# ALL pathways ordered by subtype effect
# ==========================================================

pathway_order = (
    GSVA_Subtype_Comparison
    .sort_values(
        "Effect_Pro_vs_Met",
        ascending=True
    )["Pathway"]
    .tolist()
)

pathway_order = [
    p for p in pathway_order
    if p in GSVA_matrix.index
]


heatmap_ordered = GSVA_matrix.loc[
    pathway_order,
    sample_order
].copy()


heatmap_ordered_z = heatmap_ordered.apply(
    lambda x: (
        x - x.mean()
    ) / x.std(),
    axis=1
)


heatmap_ordered_z.index = (
    heatmap_ordered_z.index
    .str.replace(
        "Pperoxisome",
        "Peroxisome",
        regex=False
    )
)

In [ ]:
# ==========================================================
# Optimized ALL Hallmark GSVA heatmap
# Fix overlapping title/subtype labels
# ==========================================================

import matplotlib.pyplot as plt
import seaborn as sns


# ----------------------------------------------------------
# Basic numbers
# ----------------------------------------------------------

n_metabolic = (
    subtype_step14 == "Metabolic"
).sum()

n_total = len(sample_order)

n_proliferative = (
    n_total - n_metabolic
)


# ----------------------------------------------------------
# Plot
# ----------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(14, 16)
)


sns.heatmap(
    heatmap_ordered_z,

    cmap="RdBu_r",
    center=0,

    xticklabels=False,
    yticklabels=True,

    cbar_kws={
        "label": "GSVA Z-score",
        "shrink": 0.60,
        "pad": 0.03
    },

    ax=ax
)


# ----------------------------------------------------------
# Boundary between subtypes
# ----------------------------------------------------------

ax.axvline(
    n_metabolic,
    linewidth=1.8
)


# ----------------------------------------------------------
# Axis labels
# ----------------------------------------------------------

ax.set_xlabel(
    "Patients",
    fontsize=11
)

ax.set_ylabel(
    "Hallmark pathways",
    fontsize=11
)


# ----------------------------------------------------------
# Main title
# Move title upward
# ----------------------------------------------------------

ax.set_title(
    "Hallmark GSVA Landscape Ordered by Subtype Association",
    fontsize=16,
    pad=42
)


# ----------------------------------------------------------
# Subtype labels
# Put them clearly below title and above heatmap
# ----------------------------------------------------------

metabolic_center = (
    n_metabolic / 2
)

proliferative_center = (
    n_metabolic
    +
    n_proliferative / 2
)


ax.text(
    metabolic_center,
    -0.8,
    "Metabolic",
    ha="center",
    va="bottom",
    fontsize=12,
    fontweight="bold",
    clip_on=False
)


ax.text(
    proliferative_center,
    -0.8,
    "Proliferative",
    ha="center",
    va="bottom",
    fontsize=12,
    fontweight="bold",
    clip_on=False
)


# ----------------------------------------------------------
# Improve pathway labels
# ----------------------------------------------------------

ax.tick_params(
    axis="y",
    labelsize=9
)


# ----------------------------------------------------------
# Give extra space at top / left / right
# ----------------------------------------------------------

fig.subplots_adjust(
    top=0.91,
    left=0.23,
    right=0.90,
    bottom=0.06
)


# ----------------------------------------------------------
# Save BEFORE show
# ----------------------------------------------------------

fig.savefig(
    os.path.join(
        figures_dir,
        "Hallmark_GSVA_All_Pathways_Ordered_Heatmap_Optimized.png"
    ),
    dpi=600,
    bbox_inches="tight"
)


fig.savefig(
    os.path.join(
        figures_dir,
        "Hallmark_GSVA_All_Pathways_Ordered_Heatmap_Optimized.svg"
    ),
    bbox_inches="tight"
)


plt.show()

plt.close(fig)